In [ ]:
import requests
import json

## What Is an API?

An **API** (Application Programming Interface) is a defined way for two programs to communicate. A **REST API** (Representational State Transfer) is the most common style for web APIs:

- The client (your code) sends an **HTTP request** to a URL called an **endpoint**
- The server processes the request and sends back an **HTTP response** — usually JSON

Think of it like a restaurant menu: the menu lists what you can order (endpoints), you place an order (request), and the kitchen sends back your food (response).

### HTTP Methods

| Method | Typical use |
|---|---|
| `GET` | Retrieve data (read-only, safe to repeat) |
| `POST` | Create a new resource |
| `PUT` / `PATCH` | Update an existing resource |
| `DELETE` | Remove a resource |

### HTTP Status Codes

| Code | Meaning |
|---|---|
| `200 OK` | Request succeeded |
| `201 Created` | Resource was created |
| `400 Bad Request` | Client sent invalid data |
| `401 Unauthorized` | Authentication required |
| `404 Not Found` | Endpoint or resource doesn't exist |
| `500 Internal Server Error` | Something went wrong on the server |

## Making GET Requests with `requests`

The `requests` library is the standard Python tool for HTTP. Install it with `pip install requests`.

```python
response = requests.get(url)
response.status_code   # integer status code (200, 404, …)
response.headers       # dict of response headers
response.text          # response body as a string
response.json()        # response body parsed as JSON → dict or list
```

We'll use **httpbin.org** — a free service that echoes back your request, perfect for learning.

In [ ]:
# httpbin.org/get echoes back details about your GET request as JSON
response = requests.get('https://httpbin.org/get')

print("Status code:", response.status_code)          # 200
print("Content-Type:", response.headers['Content-Type'])

# Parse the JSON body into a Python dict
data = response.json()
print("\nYour IP address:", data['origin'])
print("Request URL:", data['url'])

## Query Parameters

Most APIs accept configuration via **query parameters** — key/value pairs appended to the URL after `?`. Instead of building the URL string manually, pass a `params` dict to `requests.get()`:

```python
# Manual:   https://example.com/search?q=python&page=2
# With params dict (same result, cleaner):
requests.get('https://example.com/search', params={'q': 'python', 'page': 2})
```

In [ ]:
# httpbin.org/get echoes back the params you send
params = {'course': 'IST5551', 'chapter': 15}
response = requests.get('https://httpbin.org/get', params=params)

data = response.json()
print("Full URL requested:", data['url'])
# → https://httpbin.org/get?course=IST5551&chapter=15

print("Args received by server:")
for key, value in data['args'].items():
    print(f"  {key} = {value}")

## Error Handling

Two levels of errors can occur:

1. **Network error** — no connection, timeout, DNS failure → `requests` raises an exception
2. **HTTP error** — server responded but with a 4xx or 5xx status → `response.raise_for_status()` raises `HTTPError`

```python
try:
    response = requests.get(url, timeout=5)
    response.raise_for_status()          # raises HTTPError for 4xx/5xx
    data = response.json()
except requests.exceptions.Timeout:
    print("Request timed out")
except requests.exceptions.HTTPError as e:
    print(f"HTTP error {e.response.status_code}: {e}")
except requests.exceptions.RequestException as e:
    print(f"Network error: {e}")
```

Always set a **timeout** — without one, your code can hang indefinitely.

In [ ]:
def safe_get(url, params=None):
    """Make a GET request and return the JSON data, or None on failure."""
    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Timeout requesting {url}")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP {e.response.status_code} error from {url}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    return None


# Test with a valid URL
data = safe_get('https://httpbin.org/get', params={'test': 'ok'})
if data:
    print("Success — status via raise_for_status passed")
    print("Args:", data['args'])

# Test with an invalid URL (404)
bad = safe_get('https://httpbin.org/status/404')
print("404 result:", bad)   # None

In [ ]:
### Exercise: Weather API
#   The Open-Meteo API (https://api.open-meteo.com) is free — no key required.
#   Endpoint: https://api.open-meteo.com/v1/forecast
#   Required query params:
#     latitude=<float>    longitude=<float>    current=temperature_2m,wind_speed_10m
#
#   1. Use safe_get() to fetch current weather for New York (lat=40.71, lon=-74.01).
#   2. Extract and print the current temperature (°C) and wind speed (km/h).
#      The values are inside data['current'].
#   3. Fetch weather for a second city of your choice and compare temperatures.
### Your code starts here.
BASE_URL = 'https://api.open-meteo.com/v1/forecast'


### Your code ends here.

In [ ]:
### Solution
BASE_URL = 'https://api.open-meteo.com/v1/forecast'

cities = {
    'New York':    {'latitude': 40.71, 'longitude': -74.01},
    'Los Angeles': {'latitude': 34.05, 'longitude': -118.24},
}

for city, coords in cities.items():
    params = {**coords, 'current': 'temperature_2m,wind_speed_10m'}
    data = safe_get(BASE_URL, params=params)
    if data:
        current = data['current']
        temp  = current['temperature_2m']
        wind  = current['wind_speed_10m']
        print(f"{city}: {temp}°C, wind {wind} km/h")

## POST Requests

A **GET** request retrieves data from a server. A **POST** request *sends*
data to a server — typically to create a resource or submit a form. POST
requests carry a **body** (payload), usually formatted as JSON.


In [ ]:
import requests

# httpbin.org/post echoes back whatever you send — great for testing
url = 'https://httpbin.org/post'

payload = {
    'username': 'ada',
    'score': 98,
    'passed': True
}

response = requests.post(url, json=payload, timeout=10)
response.raise_for_status()

data = response.json()
print('Status:', response.status_code)   # 200
print('JSON sent:', data['json'])         # echoes our payload


## Headers and API Keys

Many APIs require authentication. Common patterns:

| Pattern | Example |
|---|---|
| Query parameter | `?api_key=abc123` |
| Bearer token header | `Authorization: Bearer <token>` |
| Custom header | `X-API-Key: abc123` |

Pass headers as a dictionary to the `headers=` argument.
**Never hard-code real API keys in code**; use environment variables instead.


In [ ]:
import requests, os

# Pattern 1: Bearer token (e.g., GitHub, OpenAI)
# token = os.environ['GITHUB_TOKEN']  # load from environment in real code
token = 'demo_token_placeholder'

headers = {
    'Authorization': f'Bearer {token}',
    'Accept': 'application/json',
}

# httpbin.org/headers echoes received headers
r = requests.get('https://httpbin.org/headers', headers=headers, timeout=10)
print('Headers echoed back:')
for k, v in r.json()['headers'].items():
    print(f'  {k}: {v}')


## Navigating Nested JSON

Real-world API responses are often deeply nested. You can chain `[]`
subscripts and use `.get()` to safely navigate the structure.


In [ ]:
import requests

# Open-Meteo: free weather API (no key required)
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 37.77,
    'longitude': -122.42,
    'current_weather': True,
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()
data = r.json()

# Navigate nested dict
weather = data['current_weather']
print(f"Temperature: {weather['temperature']} °C")
print(f"Wind speed:  {weather['windspeed']} km/h")
print(f"Weather code: {weather['weathercode']}")


In [ ]:
# Safe navigation with .get() for optional keys
sample = {
    'user': {
        'profile': {
            'name': 'Alice',
            'email': 'alice@example.com'
        }
    }
}

# Chained .get() returns None instead of raising KeyError
name  = sample.get('user', {}).get('profile', {}).get('name', 'Unknown')
phone = sample.get('user', {}).get('profile', {}).get('phone', 'Not provided')

print(name)   # Alice
print(phone)  # Not provided


In [ ]:
### Exercise: Working with APIs
#   Use the Open-Meteo API (no key needed) to answer:
#   1. Make a GET request with latitude=40.71 (New York) and longitude=-74.01.
#      Include current_weather=True in params.
#   2. Extract and print the temperature and wind speed.
#   3. Make a POST request to https://httpbin.org/post with a JSON body
#      containing {'city': 'New York', 'temp': <the temperature you got>}.
#      Print the 'json' field from the response to confirm what was sent.
import requests

### Your code starts here.



### Your code ends here.


In [ ]:
### Solution
import requests

# 1 & 2: GET weather for New York
r = requests.get(
    'https://api.open-meteo.com/v1/forecast',
    params={'latitude': 40.71, 'longitude': -74.01, 'current_weather': True},
    timeout=10
)
r.raise_for_status()
weather = r.json()['current_weather']
temp = weather['temperature']
wind = weather['windspeed']
print(f'New York — Temp: {temp} °C, Wind: {wind} km/h')

# 3: POST with the temperature
resp = requests.post(
    'https://httpbin.org/post',
    json={'city': 'New York', 'temp': temp},
    timeout=10
)
resp.raise_for_status()
print('Sent:', resp.json()['json'])


## Summary

| Concept | Key idea |
|---|---|
| REST API | URL endpoints that accept HTTP requests and return data (usually JSON) |
| `requests.get(url)` | Make a GET request; returns a `Response` object |
| `response.status_code` | Integer HTTP status (200 = OK, 404 = Not Found, …) |
| `response.json()` | Parse the JSON body into a Python dict or list |
| `params={}` | Pass query parameters cleanly — `requests` builds the URL string |
| `response.raise_for_status()` | Raises `HTTPError` on 4xx/5xx responses |
| `timeout=5` | Always set a timeout to avoid hanging indefinitely |
| `requests.exceptions.RequestException` | Base class for all `requests` errors |